Data generation for Power BI BIM (Budget Impact Model)


In [ ]:
import pandas as pd
import numpy as np
import math

# Generate 10,000 samples 
sample = 10000
patient_id = np.arange(1,sample+1,dtype=int) # 1st COLUMN


# Weights are Czech averages ( in kg, from UZIS page)
weight_men_kg = 83.6
weight_women_kg = 69.2

# variance is not known, values are estimated
var_men = 13.88**2
var_women = 15.49**2


# for lognormal function 
mu_men = np.log(weight_men_kg)
mu_women= np.log(weight_women_kg)
 
sigma_men = np.sqrt(np.log(1 + var_men / weight_men_kg**2))
sigma_women = np.sqrt(np.log(1 + var_women / weight_women_kg**2))



# Does the urothelial cancer occur more often in women or men?
# ÚZIS ČR / Uroweb – epidemiology of urogenital cancers in the Czech Republic (NOR data, C67, 1 559 men and 540 women)
probability_of_men = 0.7427

rng = np.random.default_rng(42)
is_man = rng.random(sample) < probability_of_men


sex = np.where(
    is_man,
    "Male",
    "Female",
)# 2nd COLUMN


weights = np.where(
    is_man,
    np.round(rng.lognormal(mean=mu_men,   sigma=sigma_men,   size=sample),2),
    np.round(rng.lognormal(mean=mu_women, sigma=sigma_women, size=sample),2),
)# 3rd COLUMN



# dose that was given to the patient
dispense_dose = np.minimum(
   weights * 1.25,
    125
)

# dose that was charged = dose that was given to the patient + waste
charged_dose = np.minimum(
    np.ceil((weights * 1.25) / 10) * 10,
    130
)

mean_of_required_dose = np.mean(dispense_dose)

waste = charged_dose -dispense_dose


In [2]:
columns = {"Id": patient_id , "Weight": weights,"Sex": sex ,"Dispense dose": dispense_dose ,"Charged dose": charged_dose ,"Waste": waste}

In [3]:
df_patients = pd.DataFrame(data = columns)

In [ ]:
df_patients.to_csv("patients_data.csv", index=False, sep=";", decimal=".")